# Write Delta tables to GCS from Spark

Spark writes Delta tables to a real GCS bucket via `gs://`; `delta-explain` runs on the **host**
and reads the same tables. Same data, two layouts — the partitioned one prunes well, the flat one
doesn't.

Prereqs: `cp .env.example .env` (set `GCS_BUCKET` and `GOOGLE_SERVICE_ACCOUNT`), then
`docker compose up -d`. The key is mounted at `/home/jovyan/key.json`.

On the host, after each write:
```
delta-explain gs://$GCS_BUCKET/lake/users \
  --option service_account="$GOOGLE_SERVICE_ACCOUNT" \
  -w "country = 'DE' AND age > 55" --min-pruning 50
```

In [ ]:
import os
from pyspark.sql import SparkSession

BUCKET = os.environ["GCS_BUCKET"]
KEYFILE = "/home/jovyan/key.json"  # mounted by docker-compose

spark = (
    SparkSession.builder.appName("delta-gcs")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.2.0,"
            "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.21")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # --- GCS connector (the GCS equivalent of hadoop-aws/s3a) ---
    .config("spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", KEYFILE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "-> gs://" + BUCKET)

In [ ]:
from pyspark.sql import functions as F

N = 60000
df = (
    spark.range(N)
    .withColumn("age", (F.rand(seed=42) * 52 + 18).cast("int"))
    .withColumn("_c", (F.rand(seed=7) * 3).cast("int"))
    .withColumn("country",
                F.when(F.col("_c") == 0, "DE").when(F.col("_c") == 1, "US").otherwise("IT"))
    .withColumn("score", F.round(F.rand(seed=2) * 39 + 60, 1))
    .withColumnRenamed("id", "uid")
    .drop("_c")
).cache()

TABLE = f"gs://{BUCKET}/lake/users"
print("rows:", df.count())

## Step 1 — a healthy write
Partitioned by `country`, sorted into narrow age bands so data skipping can rule files out.

In [ ]:
(
    df.orderBy("country", "age")
    .write.format("delta")
    .partitionBy("country")
    .option("maxRecordsPerFile", 4000)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .save(TABLE)
)
print("healthy layout written to", TABLE)

### 👉 On the host now
```
delta-explain gs://$GCS_BUCKET/lake/users \
  --option service_account="$GOOGLE_SERVICE_ACCOUNT" \
  -w "country = 'DE' AND age > 55" --min-pruning 50
```
Expect strong pruning and **exit 0**.

## Step 2 — a careless rewrite (the regression)
No partitioning, no sort — every file now spans everything, so nothing can be skipped.

In [ ]:
(
    df.repartition(6)
    .write.format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .save(TABLE)
)
print("flat/shuffled layout written to", TABLE)

### 👉 On the host again — same command
Pruning collapses toward **0%** and the gate **fails, exit 1**. A notebook just made every scan
read the whole table — no error, only the gate caught it.